In [11]:
import os

In [12]:
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [39]:
exmaple_path = "C:/Users/gorab/.cache/kagglehub/datasets/fhnw-i4ds/sdobenchmark/versions/4/SDOBenchmark_example"
with open(os.path.join(exmaple_path, "README.txt"), "r") as f:
    print(f.read())

Noisy images

Some raw data from the satellite is flagged as noisy, e.g. because of a moon eclipse or because of a recalibration. The images were still created, yet they have a string "flagged" in the ImageDescription metadata (EXIF).


File Structure

The folder layout is irrelevant for training, because each sample's id in 'meta_data.csv' maps to a sample.
See e.g. https://github.com/i4ds/SDOBenchmark/blob/master/notebooks/utils/keras_generator.py#L61
Except if you want to split the training set further, e.g. for cross validation: Make the splitting on the first layer of folders ('11402', '13386', ...).

If you still want to know more:
The folders "test" and "training" contain folders with numbers like '11402' or '13386'. Those numbers are Active Region numbers, i.e. enumarations for patches of the sun where we take samples from.
Within those folders are the sample folders. Each sample folder's name is a combination of a date and a number, e.g. '2014_01_28_12_50_00_1'. The date defin

In [65]:
train_example = os.path.join(exmaple_path, "training")

full_path = "C:/Users/gorab/.cache/kagglehub/datasets/fhnw-i4ds/sdobenchmark/versions/4/SDOBenchmark_full"

train_full = os.path.join(full_path, "training")

In [66]:
print(train_example)
print(os.listdir(train_example)[:5])

C:/Users/gorab/.cache/kagglehub/datasets/fhnw-i4ds/sdobenchmark/versions/4/SDOBenchmark_example\training
['11388', '11389', '11390', '11391', '11392']


In [67]:
first_region = os.listdir(train_example)[0]
region_path = os.path.join(train_example, first_region)

print(region_path)
print(os.listdir(region_path)[:5])

C:/Users/gorab/.cache/kagglehub/datasets/fhnw-i4ds/sdobenchmark/versions/4/SDOBenchmark_example\training\11388
['2012_01_07_02_27_01_0']


In [68]:
first_time = os.listdir(region_path)[0]
time_path = os.path.join(region_path, first_time)

print(time_path)
print(os.listdir(time_path))

C:/Users/gorab/.cache/kagglehub/datasets/fhnw-i4ds/sdobenchmark/versions/4/SDOBenchmark_example\training\11388\2012_01_07_02_27_01_0
['2012-01-06T142701__131.jpg', '2012-01-06T142701__1700.jpg', '2012-01-06T142701__171.jpg', '2012-01-06T142701__193.jpg', '2012-01-06T142701__211.jpg', '2012-01-06T142701__304.jpg', '2012-01-06T142701__335.jpg', '2012-01-06T142701__94.jpg', '2012-01-06T142701__continuum.jpg', '2012-01-06T142701__magnetogram.jpg', '2012-01-06T212701__131.jpg', '2012-01-06T212701__1700.jpg', '2012-01-06T212701__171.jpg', '2012-01-06T212701__193.jpg', '2012-01-06T212701__211.jpg', '2012-01-06T212701__304.jpg', '2012-01-06T212701__335.jpg', '2012-01-06T212701__94.jpg', '2012-01-06T212701__continuum.jpg', '2012-01-06T212701__magnetogram.jpg', '2012-01-07T005701__131.jpg', '2012-01-07T005701__1700.jpg', '2012-01-07T005701__171.jpg', '2012-01-07T005701__193.jpg', '2012-01-07T005701__211.jpg', '2012-01-07T005701__304.jpg', '2012-01-07T005701__335.jpg', '2012-01-07T005701__94.jpg'

In [123]:
from torch.utils.data import Dataset
from PIL import Image
import os

class SolarDataset(Dataset):
    def __init__(self, root, transform=None, max_regions=100000):
        self.samples = []
        self.transform = transform

        regions = os.listdir(root)[:max_regions]
        total_times = 0;

        for region in regions:
            region_path = os.path.join(root, region)

            if not os.path.isdir(region_path):
                continue

            times = os.listdir(region_path)
            total_times += len(times)


            for t in times[:1]:  # one timestamp group
                time_path = os.path.join(region_path, t)
                if not os.path.isdir(time_path):
                    continue

                for file in os.listdir(time_path):
                    if file.endswith("magnetogram.jpg"):
                        self.samples.append(os.path.join(time_path, file))
        
        print("TOTAL MAGNETOGRAM SAMPLES:", len(self.samples))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img = Image.open(self.samples[idx]).convert("L")

        if self.transform:
            img = self.transform(img)

        return img

In [124]:
dataset_example = SolarDataset(train_example)
print("Processing train_example ...")
print(len(dataset_example))
print(dataset_example[0])

TOTAL MAGNETOGRAM SAMPLES: 1809
Processing train_example ...
1809
<PIL.Image.Image image mode=L size=256x256 at 0x27F493B6B10>


In [125]:
print("Processing train_full ...")
dataset_full = SolarDataset(train_full)
print(len(dataset_full))
print(dataset_full[0])

Processing train_full ...
TOTAL MAGNETOGRAM SAMPLES: 1628
1628
<PIL.Image.Image image mode=L size=256x256 at 0x27F4A84DD50>
